# 3D Manifold Vector Fields

Playground notebook for S-curve, Swiss roll, and half-sphere vector-field generation.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
while not (ROOT / "simulation").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from simulation.manifold_velocity_flows import (
    MANIFOLD_NAMES,
    ManifoldVelocityFlowConfig,
    make_all_manifold_velocity_flows,
    make_manifold_velocity_flow,
    save_npz,
)

MANIFOLD_NAMES

In [ ]:
def plot_3d_flow(simulation, ax=None, max_arrows=300, seed=42):
    if ax is None:
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(111, projection="3d")

    rng = np.random.default_rng(seed)
    x = simulation["X"][:, :3]
    v = simulation["V"][:, :3]
    time = simulation["true_time"]
    time = (time - time.min()) / (time.max() - time.min() + 1e-12)

    n_arrows = min(max_arrows, len(x))
    idx = rng.choice(len(x), size=n_arrows, replace=False)

    ax.scatter(x[:, 0], x[:, 1], x[:, 2], c=time, cmap="viridis", s=8, alpha=0.35)
    ax.quiver(
        x[idx, 0], x[idx, 1], x[idx, 2],
        v[idx, 0], v[idx, 1], v[idx, 2],
        length=0.12,
        normalize=True,
        color="tab:red",
        alpha=0.55,
        linewidth=0.8,
    )
    ax.set_title(simulation["config"]["simulation_name"])
    ax.set_axis_off()
    ax.view_init(elev=24, azim=-62)
    return ax

## One Simulation

In [ ]:
config = ManifoldVelocityFlowConfig(
    manifold_name="half_sphere",
    field_name="rotation",
    n_samples=1000,
    position_noise=0.1,
    velocity_noise=0.1,
    extra_dims=5,
    seed=42,
)

simulation = make_manifold_velocity_flow(config)
simulation["config"]

In [ ]:
plot_3d_flow(simulation)
plt.show()

## All Manifolds

In [ ]:
simulations = make_all_manifold_velocity_flows(
    n_samples=1000,
    position_noise=0.1,
    velocity_noise=0.1,
    extra_dims=5,
    seed=42,
)

fig = plt.figure(figsize=(12, 5))
for idx, simulation in enumerate(simulations.values(), start=1):
    ax = fig.add_subplot(1, len(simulations), idx, projection="3d")
    plot_3d_flow(simulation, ax=ax)
plt.tight_layout()
plt.show()

## Save Data

In [ ]:
# path = save_npz(simulation)
# path